Decoder Only Plus Ollama

In [ ]:
import pandas as pd
import numpy as np
import re
import ollama
import json
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from umap import UMAP

EMBEDDING_MODEL = 'BAAI/bge-m3'
TRANSLATOR_MODEL = "qwen2.5:0.5b"
EVALUATOR_MODEL = "qwen2.5:7b" 
TARGET_IDIOM_STR = "ah la vache"

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'[^\w\s\-\'àâçéèêëîïôûùüÿñæœ]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def generate_translation(text, model):
    prompt = f"French: {text}\nEnglish:"
    try:
        response = ollama.generate(model=model, prompt=prompt)
        return response['response'].strip()
    except Exception as e:
        return ""

def evaluate_translation_category(source, reference, candidate, idiom, model):
    """Juge la qualité de la traduction selon la taxonomie IDIOMEVAL"""
    prompt = f"""
    ## Task
    Analyze the French idiom "{idiom}", the source sentence, the Reference translation, and the Candidate translation.
    Classify the Candidate translation into ONE category.
    
    ## Categories
    - Good Translation: Correct figurative meaning.
    - Literal Translation: Word-for-word, ignores figurative meaning.
    - Mistranslation: Wrong meaning.
    - Partial Translation: Misses part of the meaning.
    - Unnatural: Correct meaning but awkward phrasing.
    - Hallucination: Completely unrelated content.

    ## Input
    * Source: "{source}"
    * Reference: "{reference}"
    * Candidate: "{candidate}"

    Respond ONLY with a JSON object: {{"category": "Category Name"}}
    """
    try:
        response = ollama.generate(model=model, prompt=prompt, format='json')
        return json.loads(response['response']).get('category', 'Unknown')
    except:
        return "Evaluation Error"

print("Chargement des données...")
try:
    df = pd.read_csv('../large_idiom_set_fr_eng.csv')
except FileNotFoundError:
    # Données factices pour l'exemple
    df = pd.DataFrame({
        'original_text': ["Il faut aller droit au but.", "Arrête de tourner autour du pot, va droit au but."],
        'text': ["We must get straight to the point.", "Stop beating around the bush, get to the point."],
        'contains_idioms': ["aller droit à le but", "aller droit à le but"]
    })

df['clean_french'] = df['original_text'].apply(clean_text)
df['clean_english'] = df['text'].apply(clean_text)

# Filtrage sur l'idiome cible
if 'contains_idioms' in df.columns:
    df_target = df[df['contains_idioms'].astype(str).str.contains(TARGET_IDIOM_STR, na=False)].copy()
else:
    target_clean = clean_text(TARGET_IDIOM_STR)
    df_target = df[df['clean_french'].str.contains(target_clean, regex=False)].copy()

print(f"Traitement de {len(df_target)} phrases contenant '{TARGET_IDIOM_STR}'")
df_target = df_target.reset_index(drop=True)




Chargement des données...
Traitement de 10 phrases contenant 'ah la vache'


In [6]:

print(f"\n--- Génération ({TRANSLATOR_MODEL}) et Évaluation ({EVALUATOR_MODEL}) ---")
tqdm.pandas()

# A. Génération
df_target['generated_translation'] = df_target['clean_french'].progress_apply(
    lambda x: generate_translation(x, TRANSLATOR_MODEL)
)

# B. Évaluation (Juge)
df_target['eval_category'] = df_target.progress_apply(
    lambda row: evaluate_translation_category(
        row['clean_french'], 
        row['clean_english'], 
        row['generated_translation'], 
        TARGET_IDIOM_STR, 
        EVALUATOR_MODEL
    ), axis=1
)

print("Distribution des erreurs détectées :")
print(df_target['eval_category'].value_counts())




print(f"\nChargement du modèle d'embedding {EMBEDDING_MODEL}...")
model = SentenceTransformer(EMBEDDING_MODEL)
data_records = []

# A. Encoder les Références Anglaises (Vérité)
print("Encodage des Références Anglaises...")
eng_embeddings = model.encode(df_target['clean_english'].tolist(), show_progress_bar=True)
for idx, (text, emb) in enumerate(zip(df_target['clean_english'], eng_embeddings)):
    data_records.append({
        'idiom_id': idx,
        'text': text,
        'point_type': 'English Reference (Gold)',
        'category': 'Reference', # Couleur fixe pour la référence
        'embedding': emb
    })

# B. Encoder les Traductions Générées (Candidats)
print("Encodage des Traductions Générées...")
gen_embeddings = model.encode(df_target['generated_translation'].tolist(), show_progress_bar=True)
for idx, (text, emb, cat) in enumerate(zip(df_target['generated_translation'], gen_embeddings, df_target['eval_category'])):
    data_records.append({
        'idiom_id': idx,
        'text': text,
        'point_type': 'Generated Translation',
        'category': cat, # Sera "Literal", "Good", etc. (Couleur variable selon la qualité)
        'embedding': emb
    })

# C. Encoder la Source Française (Original uniquement)
print("Encodage de la Source Française...")
fr_embeddings = model.encode(df_target['clean_french'].tolist(), show_progress_bar=True)
for idx, (text, emb) in enumerate(zip(df_target['clean_french'], fr_embeddings)):
    data_records.append({
        'idiom_id': idx,
        'text': text,
        'point_type': 'French Source',
        'category': 'Source', # Couleur fixe pour la source
        'embedding': emb
    })

df_results = pd.DataFrame(data_records)




--- Génération (qwen2.5:0.5b) et Évaluation (qwen2.5:7b) ---


100%|██████████| 10/10 [00:12<00:00,  1.25s/it]


Distribution des erreurs détectées :
eval_category
Literal Translation    5
Mistranslation         4
Hallucination          1
Name: count, dtype: int64

Chargement du modèle d'embedding BAAI/bge-m3...
Encodage des Références Anglaises...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Encodage des Traductions Générées...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Encodage de la Source Française...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [7]:

print("\nProjection UMAP...")
if len(df_results) > 5:
    all_embeddings = np.stack(df_results['embedding'].values)
    reducer = UMAP(n_components=2, metric='cosine', random_state=42)
    projections = reducer.fit_transform(all_embeddings)
    
    df_results['x'] = projections[:, 0]
    df_results['y'] = projections[:, 1]
    
    export_df = df_results.drop(columns=['embedding']).copy()
    
    parquet_name = 'idiom_eval_analysis_bgem3.parquet'
    csv_name = 'idiom_eval_analysis_viz.csv'
    
    df_results.to_parquet(parquet_name)
    export_df.to_csv(csv_name, index=False)
    
    print(f"\nTerminé ! Fichiers générés : {parquet_name} et {csv_name}")
else:
    print("Pas assez de données pour UMAP.")


Projection UMAP...

Terminé ! Fichiers générés : idiom_eval_analysis_bgem3.parquet et idiom_eval_analysis_viz.csv


C:\Users\benja\AppData\Roaming\Python\Python312\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In Terminal : embedding-atlas Code2Qwen/embeddings_full.parquet --x x --y y --text text

In [8]:
from sklearn.metrics.pairwise import cosine_similarity

# Extract embeddings by point type for easier access
french_embeddings = df_results[df_results['point_type'] == 'French Source'].reset_index(drop=True)
english_embeddings = df_results[df_results['point_type'] == 'English Reference (Gold)'].reset_index(drop=True)
generated_embeddings = df_results[df_results['point_type'] == 'Generated Translation'].reset_index(drop=True)

# Compute cosine similarity scores
similarity_results = []

for idx in range(len(french_embeddings)):
    french_emb = french_embeddings.loc[idx, 'embedding'].reshape(1, -1)
    
    # Similarity: French → English Reference
    eng_emb = english_embeddings.loc[idx, 'embedding'].reshape(1, -1)
    sim_fr_eng = cosine_similarity(french_emb, eng_emb)[0][0]
    
    # Similarity: French → Generated Translation
    gen_emb = generated_embeddings.loc[idx, 'embedding'].reshape(1, -1)
    sim_fr_gen = cosine_similarity(french_emb, gen_emb)[0][0]
    
    similarity_results.append({
        'idiom_id': idx,
        'french_text': french_embeddings.loc[idx, 'text'],
        'english_reference': english_embeddings.loc[idx, 'text'],
        'generated_translation': generated_embeddings.loc[idx, 'text'],
        'eval_category': generated_embeddings.loc[idx, 'category'],
        'similarity_french_to_english': sim_fr_eng,
        'similarity_french_to_generated': sim_fr_gen,
        'similarity_gap': sim_fr_eng - sim_fr_gen  # How much the generated differs from the reference
    })

df_similarity = pd.DataFrame(similarity_results)
print("\nCosine Similarity Scores (French source vs. translations):")
print(df_similarity)

# Optional: Save to CSV
df_similarity.to_csv('idiom_similarity_scores.csv', index=False)
print("\nSaved to idiom_similarity_scores.csv")


Cosine Similarity Scores (French source vs. translations):
   idiom_id                                        french_text  \
0         0  ils avaient des brosses impeccables - ah la vache   
1         1                        ah la vache il l'a pas volé   
2         2                                        ah la vache   
3         3        salut ah la vache ils sont costauds ceux-là   
4         4                                        ah la vache   
5         5                                        ah la vache   
6         6                        ah la vache c'est le saumon   
7         7                    ah la vache on a un fax de l'ij   
8         8                                        ah la vache   
9         9                    - d'accord d'accord ah la vache   

                english_reference  \
0         - and your grandfathers   
1           he sure had it coming   
2                     bloody hell   
3  oh my god who's winning so far   
4                     - holy